# Train leaf instance-segmentation candidates

Notebook này chứa training loop thật cho các YOLO segmentation candidate. DVC thực thi notebook bằng Papermill; notebook ghi checkpoint và metrics theo contract mà `pipeline.py select` sử dụng.

Sau khi train, notebook đọc `results.csv` của Ultralytics và lưu các biểu đồ báo cáo dưới `metrics/` (training curves, validation metrics, model comparison, prediction overlays và sample masks).

In [ ]:
params_path = "params.yaml"

In [ ]:
from pathlib import Path
from typing import Any
import csv
import json
import os
import shutil
import tempfile

project_root = Path.cwd().resolve()
if not (project_root / "pipeline.py").exists():
    project_root = project_root.parent
if not (project_root / "pipeline.py").exists():
    raise RuntimeError("Run this notebook from the CoffeeLeaf-AI repository")
os.chdir(project_root)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import display
from PIL import Image, ImageDraw
from ultralytics import YOLO

from pipeline import (
    ROOT,
    load_config,
    project_path,
    reset_dir,
    seed_everything,
    write_csv,
    write_json,
)

print(f"Repository: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
PLOT_DPI = 300


def nested_attr(obj: Any, path: str, default: float = 0.0) -> float:
    current = obj
    for name in path.split("."):
        if current is None:
            return default
        current = getattr(current, name, None)
    try:
        return float(current)
    except (TypeError, ValueError):
        return default


def runtime_yolo_yaml(segment_root: Path) -> Path:
    payload = {
        "path": str(segment_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "leaf"},
    }
    with tempfile.NamedTemporaryFile(
        "w", suffix=".yaml", encoding="utf-8", delete=False
    ) as handle:
        yaml.safe_dump(payload, handle, sort_keys=False)
        return Path(handle.name)


def resolve_device(value: Any) -> Any:
    return None if str(value).lower() == "auto" else value


def _normalize_header(name: str) -> str:
    return "".join(ch for ch in name.strip().lower() if ch.isalnum() or ch in "/()-_")


def find_result_column(columns: list[str], *candidates: str) -> str | None:
    normalized = {_normalize_header(column): column for column in columns}
    for candidate in candidates:
        key = _normalize_header(candidate)
        if key in normalized:
            return normalized[key]
    for candidate in candidates:
        needle = _normalize_header(candidate)
        for key, original in normalized.items():
            if needle in key:
                return original
    return None


def read_ultralytics_results_csv(path: Path) -> dict[str, list[float]]:
    """Load Ultralytics results.csv into column -> numeric series."""
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"Empty results.csv: {path}")
        original_columns = list(reader.fieldnames)
        columns = [name.strip() for name in original_columns]
        series: dict[str, list[float]] = {name: [] for name in columns}
        for row in reader:
            for original, name in zip(original_columns, columns):
                raw = (row.get(original) or "").strip()
                try:
                    series[name].append(float(raw))
                except ValueError:
                    series[name].append(float("nan"))
    return series


def count_model_parameters(model: Any) -> int:
    underlying = getattr(model, "model", model)
    return int(sum(parameter.numel() for parameter in underlying.parameters()))


def load_yolo_seg_mask(label_path: Path, height: int, width: int) -> np.ndarray:
    mask = Image.new("L", (width, height), 0)
    drawer = ImageDraw.Draw(mask)
    if not label_path.exists():
        return np.asarray(mask, dtype=np.uint8)
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        coords = np.asarray(list(map(float, parts[1:])), dtype=np.float32).reshape(-1, 2)
        pixels = [
            (float(x) * width, float(y) * height) for x, y in coords.tolist()
        ]
        drawer.polygon(pixels, outline=255, fill=255)
    return np.asarray(mask, dtype=np.uint8)


def prediction_mask_from_result(result: Any, height: int, width: int) -> np.ndarray:
    masks = getattr(result, "masks", None)
    if masks is None or getattr(masks, "data", None) is None or len(masks.data) == 0:
        return np.zeros((height, width), dtype=np.uint8)
    stacked = masks.data.detach().cpu().numpy() > 0.5
    combined = stacked.any(axis=0).astype(np.uint8) * 255
    if combined.shape != (height, width):
        resized = Image.fromarray(combined).resize((width, height), Image.NEAREST)
        return np.asarray(resized, dtype=np.uint8)
    return combined


def overlay_mask(
    image: np.ndarray, mask: np.ndarray, color: tuple[int, int, int] = (46, 204, 113), alpha: float = 0.45
) -> np.ndarray:
    base = image.astype(np.float32).copy()
    active = mask > 0
    if not np.any(active):
        return base.astype(np.uint8)
    tint = np.zeros_like(base)
    tint[active] = np.asarray(color, dtype=np.float32)
    blended = base.copy()
    blended[active] = (1.0 - alpha) * base[active] + alpha * tint[active]
    return np.clip(blended, 0, 255).astype(np.uint8)


def resize_for_display(image: np.ndarray, max_side: int = 768) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(1.0, float(max_side) / float(max(height, width)))
    if scale >= 1.0:
        return image
    size = (max(1, int(round(width * scale))), max(1, int(round(height * scale))))
    mode = "L" if image.ndim == 2 else "RGB"
    resample = Image.NEAREST if image.ndim == 2 else Image.BILINEAR
    return np.asarray(Image.fromarray(image, mode=mode).resize(size, resample))


def list_validation_images(images_dir: Path, limit: int) -> list[Path]:
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    paths = sorted(
        path
        for path in images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )
    if not paths:
        return []
    if len(paths) <= limit:
        return paths
    indices = np.linspace(0, len(paths) - 1, num=limit, dtype=int)
    return [paths[index] for index in indices]


def plot_segmentation_training_curves(
    histories: dict[str, dict[str, list[float]]],
    output_path: Path,
) -> Path:
    candidates = list(histories)
    fig, axes = plt.subplots(
        len(candidates), 2, figsize=(14, 4.2 * max(len(candidates), 1)), squeeze=False
    )
    train_specs = [
        ("train/box_loss", "train box loss", ("train/box_loss", "train/box")),
        ("train/seg_loss", "train seg loss", ("train/seg_loss", "train/seg")),
        ("train/cls_loss", "train cls loss", ("train/cls_loss", "train/cls")),
    ]
    val_specs = [
        ("val/box_loss", "val box loss", ("val/box_loss", "val/box")),
        ("val/seg_loss", "val seg loss", ("val/seg_loss", "val/seg")),
        ("val/cls_loss", "val cls loss", ("val/cls_loss", "val/cls")),
    ]
    for row_index, candidate in enumerate(candidates):
        series = histories[candidate]
        columns = list(series)
        epoch_column = find_result_column(columns, "epoch")
        epochs = (
            series[epoch_column]
            if epoch_column is not None
            else list(range(1, len(next(iter(series.values()))) + 1))
        )
        for axis, specs, title in (
            (axes[row_index, 0], train_specs, f"{candidate}: training losses"),
            (axes[row_index, 1], val_specs, f"{candidate}: validation losses"),
        ):
            plotted = False
            for _, label, aliases in specs:
                column = find_result_column(columns, *aliases)
                if column is None:
                    continue
                axis.plot(epochs, series[column], label=label, linewidth=2)
                plotted = True
            axis.set_title(title)
            axis.set_xlabel("Epoch")
            axis.set_ylabel("Loss")
            axis.grid(True, alpha=0.3)
            if plotted:
                axis.legend()
            else:
                axis.text(0.5, 0.5, "No loss columns found", ha="center", va="center")
    fig.suptitle("Segmentation training curves", fontsize=16, fontweight="bold")
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


def plot_segmentation_metrics(
    histories: dict[str, dict[str, list[float]]],
    output_path: Path,
) -> Path:
    metric_specs = [
        ("Precision", ("metrics/precision(M)", "metrics/precision(B)", "metrics/precision")),
        ("Recall", ("metrics/recall(M)", "metrics/recall(B)", "metrics/recall")),
        ("mAP50", ("metrics/mAP50(M)", "metrics/mAP50(B)", "metrics/mAP50")),
        ("mAP50-95", ("metrics/mAP50-95(M)", "metrics/mAP50-95(B)", "metrics/mAP50-95")),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for axis, (title, aliases) in zip(axes.flat, metric_specs):
        for candidate, series in histories.items():
            columns = list(series)
            epoch_column = find_result_column(columns, "epoch")
            epochs = (
                series[epoch_column]
                if epoch_column is not None
                else list(range(1, len(next(iter(series.values()))) + 1))
            )
            column = find_result_column(columns, *aliases)
            if column is None:
                continue
            axis.plot(epochs, series[column], label=candidate, linewidth=2)
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(title)
        axis.set_ylim(0.0, 1.05)
        axis.grid(True, alpha=0.3)
        axis.legend()
    fig.suptitle("Segmentation validation metrics", fontsize=16, fontweight="bold")
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


def plot_segmentation_model_comparison(
    metrics_by_candidate: dict[str, dict[str, Any]],
    output_path: Path,
) -> Path:
    candidate_names = list(metrics_by_candidate)
    comparison_specs = [
        ("map50_95", "mAP50-95", True),
        ("recall", "Recall", True),
        ("precision", "Precision", True),
        ("latency_ms", "Latency (ms/image)", False),
        ("size_mb", "Model size (MB)", False),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(candidate_names), 1)))
    for axis, (metric_name, title, unit_interval) in zip(axes.flat, comparison_specs):
        values = [float(metrics_by_candidate[name].get(metric_name, 0.0)) for name in candidate_names]
        bars = axis.bar(candidate_names, values, color=colors[: len(candidate_names)])
        axis.set_title(title)
        axis.set_ylabel(title)
        axis.tick_params(axis="x", rotation=20)
        axis.grid(True, axis="y", alpha=0.3)
        if unit_interval:
            axis.set_ylim(0.0, 1.05)
        for bar, value in zip(bars, values):
            label = f"{value:.3f}" if unit_interval else f"{value:.2f}"
            axis.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                label,
                ha="center",
                va="bottom",
                fontsize=9,
            )
    axes.flat[-1].axis("off")
    fig.suptitle("Segmentation candidate comparison", fontsize=16, fontweight="bold")
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


def collect_prediction_panels(
    model: Any,
    image_paths: list[Path],
    labels_dir: Path,
    image_size: int,
) -> list[dict[str, np.ndarray]]:
    panels: list[dict[str, np.ndarray]] = []
    for image_path in image_paths:
        with Image.open(image_path) as handle:
            rgb = np.asarray(handle.convert("RGB"))
        height, width = rgb.shape[:2]
        label_path = labels_dir / f"{image_path.stem}.txt"
        ground_truth = load_yolo_seg_mask(label_path, height, width)
        prediction = model.predict(
            source=str(image_path),
            imgsz=image_size,
            verbose=False,
            save=False,
        )[0]
        pred_mask = prediction_mask_from_result(prediction, height, width)
        overlay = overlay_mask(rgb, pred_mask)
        panels.append(
            {
                "original": resize_for_display(rgb),
                "ground_truth": resize_for_display(ground_truth),
                "prediction": resize_for_display(pred_mask),
                "overlay": resize_for_display(overlay),
            }
        )
    return panels


def plot_segmentation_predictions(
    panels: list[dict[str, np.ndarray]],
    output_path: Path,
    candidate: str,
) -> Path:
    if not panels:
        raise RuntimeError("No validation images available for prediction visualization")
    rows = len(panels)
    fig, axes = plt.subplots(rows, 4, figsize=(14, 3.1 * rows), squeeze=False)
    titles = ("Original", "Ground truth", "Prediction", "Overlay")
    keys = ("original", "ground_truth", "prediction", "overlay")
    cmaps = (None, "gray", "gray", None)
    for row_index, panel in enumerate(panels):
        for column_index, (title, key, cmap) in enumerate(zip(titles, keys, cmaps)):
            axis = axes[row_index, column_index]
            image = panel[key]
            if cmap is None:
                axis.imshow(image)
            else:
                axis.imshow(image, cmap=cmap, vmin=0, vmax=255)
            if row_index == 0:
                axis.set_title(title)
            axis.set_xticks([])
            axis.set_yticks([])
            if column_index == 0:
                axis.set_ylabel(f"#{row_index + 1}", rotation=0, labelpad=18, va="center")
    fig.suptitle(
        f"Segmentation predictions ({candidate})",
        fontsize=16,
        fontweight="bold",
    )
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


def plot_segmentation_masks(
    panels: list[dict[str, np.ndarray]],
    output_path: Path,
    candidate: str,
) -> Path:
    if not panels:
        raise RuntimeError("No validation images available for mask visualization")
    columns = 4
    rows = int(np.ceil(len(panels) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.4 * rows), squeeze=False)
    for index, axis in enumerate(axes.flat):
        if index >= len(panels):
            axis.axis("off")
            continue
        panel = panels[index]
        composed = np.concatenate(
            [
                np.stack([panel["ground_truth"]] * 3, axis=-1),
                np.stack([panel["prediction"]] * 3, axis=-1),
                panel["overlay"],
            ],
            axis=1,
        )
        axis.imshow(composed)
        axis.set_title(f"GT | Pred | Overlay #{index + 1}", fontsize=10)
        axis.set_xticks([])
        axis.set_yticks([])
    fig.suptitle(
        f"Segmentation sample masks ({candidate})",
        fontsize=16,
        fontweight="bold",
    )
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


In [ ]:
config = load_config(params_path)
seed = int(config["seed"])
seed_everything(seed)
settings = config["segmentation"]
worker_count = 0 if os.name == "nt" else int(settings["workers"])
segment_root = project_path(config["data"]["processed_dir"]) / "segmentation"
required_splits = [segment_root / "images" / split for split in ("train", "val", "test")]
missing = [str(path) for path in required_splits if not path.exists()]
if missing:
    raise RuntimeError(
        "Prepared segmentation data is missing. Run `dvc repro prepare`. Missing: "
        + ", ".join(missing)
    )

metrics_dir = ROOT / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps(settings, indent=2))
print(f"Ultralytics workers: {worker_count}")
print(f"Metrics directory: {metrics_dir}")


In [ ]:
output_dir = reset_dir(ROOT / "models" / "segmenters")
data_yaml = runtime_yolo_yaml(segment_root)
metrics_by_candidate: dict[str, dict[str, Any]] = {}
rows: list[dict[str, Any]] = []
history_by_candidate: dict[str, dict[str, list[float]]] = {}
checkpoint_by_candidate: dict[str, Path] = {}

try:
    for candidate, pretrained_weights in settings["candidates"].items():
        print(f"\n=== Training segmenter: {candidate} ===")
        source = str(pretrained_weights)
        if not bool(settings.get("pretrained", True)) and source.endswith(".pt"):
            source = source[:-3] + ".yaml"
        model = YOLO(source)
        with tempfile.TemporaryDirectory(prefix=f"coffee-{candidate}-") as run_dir:
            train_args: dict[str, Any] = {
                "data": str(data_yaml),
                "epochs": int(settings["epochs"]),
                "imgsz": int(settings["image_size"]),
                "batch": int(settings["batch_size"]),
                "patience": int(settings["patience"]),
                "workers": worker_count,
                "seed": seed,
                "deterministic": bool(settings["deterministic"]),
                "project": run_dir,
                "name": candidate,
                "exist_ok": True,
                "verbose": True,
            }
            device = resolve_device(settings.get("device", "auto"))
            if device is not None:
                train_args["device"] = device
            model.train(**train_args)
            best_path = Path(model.trainer.best)
            if not best_path.exists():
                raise RuntimeError(f"Ultralytics did not create best weights for {candidate}")
            target = output_dir / f"{candidate}.pt"
            shutil.copy2(best_path, target)

            results_csv = Path(model.trainer.save_dir) / "results.csv"
            if results_csv.exists():
                preserved_csv = output_dir / f"{candidate}_results.csv"
                shutil.copy2(results_csv, preserved_csv)
                history_by_candidate[candidate] = read_ultralytics_results_csv(preserved_csv)
            else:
                print(f"Warning: Ultralytics results.csv missing for {candidate}")

        best_model = YOLO(str(target))
        validation = best_model.val(
            data=str(data_yaml),
            split="val",
            imgsz=int(settings["image_size"]),
            batch=int(settings["batch_size"]),
            workers=worker_count,
            verbose=False,
        )
        test_result = best_model.val(
            data=str(data_yaml),
            split="test",
            imgsz=int(settings["image_size"]),
            batch=int(settings["batch_size"]),
            workers=worker_count,
            verbose=False,
        )
        speed = getattr(validation, "speed", {}) or {}
        latency_ms = float(speed.get("inference", 0.0))
        size_mb = target.stat().st_size / (1024 * 1024)
        number_of_parameters = count_model_parameters(best_model)
        values = {
            "map50_95": nested_attr(validation, "seg.map"),
            "map50": nested_attr(validation, "seg.map50"),
            "precision": nested_attr(validation, "seg.mp"),
            "recall": nested_attr(validation, "seg.mr"),
            "test_map50_95": nested_attr(test_result, "seg.map"),
            "test_map50": nested_attr(test_result, "seg.map50"),
            "test_precision": nested_attr(test_result, "seg.mp"),
            "test_recall": nested_attr(test_result, "seg.mr"),
            "latency_ms": latency_ms,
            "inference_time": latency_ms,
            "size_mb": size_mb,
            "model_size_mb": size_mb,
            "number_of_parameters": number_of_parameters,
            "image_size": int(settings["image_size"]),
        }
        metrics_by_candidate[candidate] = values
        checkpoint_by_candidate[candidate] = target
        rows.append(
            {
                "candidate": candidate,
                **{
                    key: values[key]
                    for key in (
                        "map50_95",
                        "map50",
                        "precision",
                        "recall",
                        "latency_ms",
                        "size_mb",
                        "number_of_parameters",
                    )
                },
            }
        )
        del model, best_model, validation, test_result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
finally:
    data_yaml.unlink(missing_ok=True)

write_json(metrics_dir / "segmenters.json", {"candidates": metrics_by_candidate})
write_csv(metrics_dir / "segmenters.csv", rows)


## Curves, comparison and prediction visuals

Các biểu đồ dưới đây được lưu dưới `metrics/`, tái sử dụng `results.csv` của Ultralytics và checkpoint đã chọn theo `map50_95`.

In [ ]:
if not history_by_candidate:
    raise RuntimeError("No Ultralytics results.csv histories were preserved for plotting")

training_plot_path = plot_segmentation_training_curves(
    history_by_candidate, metrics_dir / "segmentation_training_curves.png"
)
metrics_plot_path = plot_segmentation_metrics(
    history_by_candidate, metrics_dir / "segmentation_metrics.png"
)
comparison_plot_path = plot_segmentation_model_comparison(
    metrics_by_candidate, metrics_dir / "segmentation_model_comparison.png"
)

best_candidate = max(
    metrics_by_candidate,
    key=lambda name: float(metrics_by_candidate[name]["map50_95"]),
)
best_checkpoint = checkpoint_by_candidate[best_candidate]
viz_model = YOLO(str(best_checkpoint))
val_images = list_validation_images(segment_root / "images" / "val", limit=10)
prediction_panels = collect_prediction_panels(
    viz_model,
    val_images,
    segment_root / "labels" / "val",
    image_size=int(settings["image_size"]),
)
predictions_plot_path = plot_segmentation_predictions(
    prediction_panels,
    metrics_dir / "segmentation_predictions.png",
    best_candidate,
)
masks_plot_path = plot_segmentation_masks(
    prediction_panels,
    metrics_dir / "segmentation_masks.png",
    best_candidate,
)

del viz_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Best validation mAP50-95: {best_candidate}")
print(
    "Saved plots: "
    f"{training_plot_path}, {metrics_plot_path}, {comparison_plot_path}, "
    f"{predictions_plot_path}, {masks_plot_path}"
)


In [ ]:
result = json.loads((metrics_dir / "segmenters.json").read_text(encoding="utf-8"))
result
